# Eye + head gate: train and test, 5-fold CV

The gaze-only gate collapsed its two outputs onto one curve (r = 0.97 between predicted
`S_frame` and `S_gaze`) and never predicted REFIXATION. This notebook trains the next idea:
add **head motion from the IMU** and the **direction agreement between eye and head**, and
classify the four classes directly.

```
eye_speed     (9) ─► causal CNN ─► f_eye(k)  ┐
head_speed    (9) ─► causal CNN ─► f_head(k) ├─ concat per step ─► MLP ─► masked mean+max ─► 4 classes
cos(eye,head) (9) ─► causal CNN ─► f_cos(k)  ┘
```

| signal | from | meaning |
|---|---|---|
| eye_speed | `gaze_rates_window` | √(eye yaw rate² + eye pitch rate²), rad/s |
| head_speed | `imu_window` gyro, in the eye's yaw/pitch axes | √(head yaw rate² + head pitch rate²), rad/s |
| cos(eye, head) | both | +1 same direction (gaze shift), −1 eyes counter-rotating (VOR), 0 when either is still |

Loss: class-weighted cross-entropy. No frames and no DINOv2 features are used.

## What the physiology predicts

| class | eye | head | cos |
|---|---|---|---|
| STABLE | slow | still | 0 |
| REFIXATION | saccade spike | still | 0 |
| PURSUIT | moving | moving | ≈ −1 |
| TRANSITION | saccade spike | moving | +1, then −1 |

Section 4 checks this on the data **before** any training.

## Run order

1. Clone, mount
2. Check the folds, thresholds and IMU coverage
3. IMU axes: factory calibration if a VRS is available, otherwise fitted from data
4. Feature check: does each class look the way the table says?
5. **Train** the full model, 5 folds
6. **Ablations**: eye only, eye + head without cos, and the shuffle control
7. **Test**, once, against the previous gate on the same rows

Everything is written to `MyDrive/GazeVLM/runs/`, so it survives the runtime. The model has
about 5k parameters; a CPU runtime is enough, a GPU just makes it faster.

## 1. Clone, mount, set paths

The code lives on the `eyehead-gate` branch until it is merged; change `BRANCH` to `main`
after that. If `/content/GazeVLM` already exists from another notebook, the cell switches it
to this branch.

In [ ]:
import os, sys, json, time, subprocess

REPO   = '/content/GazeVLM'
URL    = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
BRANCH = 'eyehead-gate'          # set to 'main' once the branch is merged

if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('eyehead/train.py'), f'eyehead/ is not on branch {BRANCH!r}'

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
THRESH    = os.path.join(DRIVE_DIR, 'thresholds.json')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
RUN_MAIN  = os.path.join(RUNS, 'eyehead')
BASELINE  = os.path.join(DRIVE_DIR, 'release', 'gate_v1', 'test_preds.csv')
IMU_CALIB = os.path.join(DRIVE_DIR, 'imu_calib.json')
os.makedirs(RUNS, exist_ok=True)


def sh(cmd, tail=None):
    """Run a shell command and stream its output into the cell (or only the last lines)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    keep = []
    for line in p.stdout:
        if tail is None:
            print(line, end='')
        else:
            keep = (keep + [line])[-tail:]
    if tail is not None:
        print(''.join(keep), end='')
    if p.wait():
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')


print('torch', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU (fine: the model is ~5k parameters)')

## 2. Check the folds, the thresholds and IMU coverage

Rows with no usable IMU are dropped from training and validation by default
(`--missing_imu drop`), for every model including the eye-only ablation, so all variants
see the same rows. At test time every row is kept (head = 0 where the IMU is missing), so
the comparison with the previous gate is on identical rows.

In [ ]:
need = [f'train_val_fold{k}.csv' for k in range(1, 6)] + ['test.csv']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDS, f))]
assert not missing, f'missing in {FOLDS}: {missing}\nRun colab_label_quadrants.ipynb first.'
assert os.path.exists(THRESH), f'missing {THRESH}'
th = json.load(open(THRESH))
print(f"tau_frame {th['tau_frame']:.4f}   tau_gaze {th['tau_gaze']:.4f}\n")

from eyehead.features import load_rows, raw_signals, NAMES

cov = []
for f in ('train_val_fold1.csv', 'test.csv'):
    d = load_rows(os.path.join(FOLDS, f))
    E, G, M, ok = raw_signals(d)
    per_vid = pd.Series(ok).groupby(d.sequence.to_numpy()).mean()
    cov.append(dict(file=f, rows=len(d), videos=d.sequence.nunique(),
                    gaze_present=f'{100*M.any(1).mean():.1f}%',
                    imu_present=f'{100*ok.mean():.1f}%',
                    videos_without_imu=int((per_vid == 0).sum())))
    if f == 'test.csv':
        print('test class balance:')
        display(d.quad.map(NAMES).value_counts().rename('rows').to_frame()
                .assign(pct=lambda x: (100 * x.rows / len(d)).round(1)))
display(pd.DataFrame(cov).set_index('file'))
print('If imu_present is far below 100%, an older batch may have been built without IMU;')
print('those rows are dropped from training and scored with head = 0 at test.')

## 3. IMU axes: calibration or fit

The gyro is stored in the IMU's own frame, so "yaw" and "pitch" mean nothing until it is
rotated into the eye's axes (CPF: +X left, +Y up, +Z forward). Two ways:

| | how | trust |
|---|---|---|
| **calib** | Aria's factory calibration, read from one VRS by `eyehead.imu_calib` | exact (devices share one design, so one VRS serves all) |
| **fit** | least squares on the training rows, assuming the eyes mostly counter-rotate the head | good if VOR dominates, but makes the VOR sign check circular |

No VRS is needed up front. The cell gets the AEA download-links JSON the same way the other
notebooks do:

- **Option B (default):** from Drive. It checks `MyDrive/GazeVLM/aea_download_urls.json`,
  where `colab_build_csv_to_drive` keeps a copy, then `MyDrive/aea/aea_download_urls.json`.
- **Option A:** set `UPLOAD_JSON = True` to pick it from your laptop; a copy is kept on
  Drive for next time.

It then downloads the **smallest** VRS in the JSON once and writes
`MyDrive/GazeVLM/imu_calib.json`. Later runs reuse that file and skip all of this.

If the links have expired (~14 days), the download fails cleanly and training falls back to
**fit**. Re-download the JSON from projectaria.com/datasets/aea, upload it, and rerun.

In [ ]:
VRS_PATH   = None        # set this only if you already have a .vrs file somewhere
UPLOAD_JSON = False      # True -> pick aea_download_urls.json from your laptop (Option A
                         # in the other notebooks); a copy is then kept on Drive

# Option B, as in the other notebooks: the JSON already on Drive. colab_build_csv_to_drive
# keeps a copy at MyDrive/GazeVLM/; the older notebooks read MyDrive/aea/. First hit wins.
URLS_CANDIDATES = [os.path.join(DRIVE_DIR, 'aea_download_urls.json'),
                   '/content/drive/MyDrive/aea/aea_download_urls.json']

if UPLOAD_JSON and not os.path.exists(IMU_CALIB):
    import shutil
    from google.colab import files
    up = files.upload()                       # pick your aea_download_urls.json
    name = list(up.keys())[0]
    shutil.copy2(name, URLS_CANDIDATES[0])    # keep it on Drive for next time
    print('kept a copy on Drive:', URLS_CANDIDATES[0])

URLS_JSON = next((p for p in URLS_CANDIDATES if os.path.exists(p)), None)


def fetch_smallest_vrs(urls_json, dest_root='/content/raw'):
    """Download ONE VRS -- the smallest in the JSON -- and return its path, or None.

    Any recording will do: the IMU->CPF rotation is fixed by the glasses' mechanical
    design, so it is effectively the same for every device. A partial file is deleted on
    failure so a rerun does not mistake it for a complete download.
    """
    import urllib.request
    meta = json.load(open(urls_json))
    cands = [(s, v['main_vrs']) for s, v in meta['sequences'].items() if 'main_vrs' in v]
    if not cands:
        print(f'{urls_json} has no main_vrs entries')
        return None
    seq, vrs = min(cands, key=lambda kv: kv[1].get('file_size_bytes', float('inf')))
    path = os.path.join(dest_root, seq, vrs['filename'])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and os.path.getsize(path) == vrs.get('file_size_bytes', -1):
        print(f'already downloaded: {path}')
        return path
    print(f"downloading the smallest VRS: {seq}  ({vrs.get('file_size_bytes', 0)/1e9:.2f} GB) ...")
    t0 = time.time()
    try:
        urllib.request.urlretrieve(vrs['download_url'], path)
    except Exception as ex:
        if os.path.exists(path):
            os.remove(path)
        print(f'   download FAILED: {ex}')
        print('   AEA links expire after ~14 days -- re-download the JSON from')
        print('   projectaria.com/datasets/aea, set UPLOAD_JSON = True, and rerun this cell.')
        return None
    print(f'   done in {(time.time()-t0)/60:.1f} min -> {path}')
    return path


if os.path.exists(IMU_CALIB):
    print('using existing', IMU_CALIB)
    print(open(IMU_CALIB).read())
else:
    if VRS_PATH is None:
        if URLS_JSON:
            print('AEA links JSON:', URLS_JSON)
            VRS_PATH = fetch_smallest_vrs(URLS_JSON)
        else:
            print('no AEA links JSON at any of:', *URLS_CANDIDATES, sep='\n   ')
            print('set UPLOAD_JSON = True to pick it from your laptop.')
    if VRS_PATH and os.path.exists(VRS_PATH):
        sh('pip -q install projectaria-tools', tail=3)
        sh(f'python -u -m eyehead.imu_calib --vrs "{VRS_PATH}" --out "{IMU_CALIB}"')
    else:
        print('\nno VRS -> the IMU map will be FIT from each fold\'s training rows instead.')

IMU_ARGS = (f'--imu_rot calib --imu_calib "{IMU_CALIB}"' if os.path.exists(IMU_CALIB)
            else '--imu_rot fit')
print('\nIMU_ARGS =', IMU_ARGS)

## 4. Look at the features before training

If the four classes do not differ in these three signals, no model will separate them, and
it is better to know that now. Read three things:

1. **VOR sign check.** When the head turns and the eye moves smoothly, cos should be clearly
   negative. A positive median means a sign or axis in the IMU map is wrong (in **fit** mode
   it is negative by construction, so it proves nothing there).
2. **Head speed** should be low for STABLE and REFIXATION and high for PURSUIT and
   TRANSITION.
3. **Eye speed max** (the saccade proxy) should be high for REFIXATION and TRANSITION.

In [ ]:
from eyehead.features import (fit_head_map, compute_channels, vor_check, class_profile,
                              load_imu_calib)

f1 = load_rows(os.path.join(FOLDS, 'train_val_fold1.csv'))
E, G, M, ok = raw_signals(f1)
y = f1.quad.to_numpy()
tr = (f1.split == 'train').to_numpy() & ok & M.any(1)

if os.path.exists(IMU_CALIB):
    A, _ = load_imu_calib(IMU_CALIB)
    print('IMU map: calibration')
else:
    A, info = fit_head_map(E[tr], G[tr], M[tr], ok[tr])
    print(f"IMU map: fit on fold-1 training rows   VOR gain {np.round(info['vor_gain'], 2)}"
          f"   R2 {info['r2']:.3f}   ({info['n_steps']:,} steps)")
print('raw gyro -> [head_yaw, head_pitch]:')
print(np.round(A, 3))

X, H = compute_channels(E, G, M, A)
v = vor_check(E[tr], H[tr], M[tr], ok[tr])
print(f"\nVOR sign check: median cos {v['median_cos']:+.3f} over {v['n']:,} steps, "
      f"{100*v['frac_negative']:.0f}% negative   "
      f"[{'ok' if v['median_cos'] < -0.3 else 'CHECK THE IMU MAP'}]")

print('\nper-class profile, fold-1 training rows (raw units):')
prof = class_profile(X[tr], M[tr], y[tr])
display(prof.round(3))

# per-row summaries for plotting
Mr = M[tr]
eye_max  = np.where(Mr, X[tr, 0], 0).max(1)
head_avg = (X[tr, 1] * Mr).sum(1) / Mr.sum(1)
cls = y[tr]
colors = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:red', 3: 'tab:green'}

fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
for a_, vals, title in ((ax[0], eye_max, 'eye speed, max step per row (rad/s)'),
                        (ax[1], head_avg, 'head speed, mean per row (rad/s)')):
    a_.boxplot([vals[cls == c] + 1e-4 for c in range(4)], showfliers=False)
    a_.set_xticks(range(1, 5)); a_.set_xticklabels(list(NAMES.values()))
    a_.set_yscale('log'); a_.set_title(title)
act = (X[tr, 2] != 0) & Mr
for c in range(4):
    vals = X[tr, 2][act & (cls == c)[:, None]]
    ax[2].hist(vals, bins=40, range=(-1, 1), histtype='step', lw=1.6, density=True,
               color=colors[c], label=NAMES[c])
ax[2].set_title('cos(eye, head), active steps'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
idx = rng.choice(len(cls), size=min(4000, len(cls)), replace=False)
plt.figure(figsize=(7, 5.5))
for c in range(4):
    s = idx[cls[idx] == c]
    plt.scatter(head_avg[s] + 1e-3, eye_max[s] + 1e-3, s=5, alpha=.35, c=colors[c], label=NAMES[c])
plt.xscale('log'); plt.yscale('log')
plt.xlabel('head speed, mean (rad/s)'); plt.ylabel('eye speed, max step (rad/s)')
plt.title('each class should sit in its own region'); plt.legend(markerscale=3, fontsize=8)
plt.show()

## 5. Train the full model (eye + head + cos), 5 folds

Per fold, the IMU map (fit mode), the normalisation and the class weights come from that
fold's training rows only. Model selection is on validation **balanced accuracy**; the CV
numbers printed at the end are the mean over the **last 5 epochs**, not the best epoch.

In [ ]:
EPOCHS, PATIENCE, BATCH, LR = 150, 25, 256, 1e-3
COMMON = (f'--folds_dir "{FOLDS}" --thresholds "{THRESH}" {IMU_ARGS} '
          f'--epochs {EPOCHS} --patience {PATIENCE} --batch_size {BATCH} --lr {LR}')

t0 = time.time()
sh(f'python -u -m eyehead.train {COMMON} --out_dir "{RUN_MAIN}"')
print(f'\ntotal {(time.time()-t0)/60:.1f} min')

### Curves and the fold spread

In [ ]:
res = json.load(open(os.path.join(RUN_MAIN, 'cv_results.json')))
H_ = res['history']

fig, ax = plt.subplots(1, 4, figsize=(21, 4.2))
for k, h in H_.items():
    ep = [r['epoch'] for r in h]
    ax[0].plot(ep, [r['tr_loss'] for r in h], lw=1.2, label=k)
    ax[1].plot(ep, [r['bal_acc'] for r in h], lw=1.3, label=k)
    ax[2].plot(ep, [r['recall_2'] for r in h], lw=1.3, label=k)
    ax[3].plot(ep, [r['recall_1'] for r in h], lw=1.3, label=k)
ax[0].set_title('train loss (weighted CE)'); ax[0].legend(fontsize=7)
ax[1].axhline(0.25, ls='--', c='k', lw=1); ax[1].set_title('val balanced accuracy (chance 0.25)')
ax[2].set_title('val REFIXATION recall'); ax[3].set_title('val PURSUIT recall')
for a_ in ax: a_.set_xlabel('epoch')
plt.tight_layout(); plt.show()

tab = pd.DataFrame([f['last'] for f in res['folds']], index=[f"fold {f['fold']}" for f in res['folds']])
tab = tab.rename(columns={'recall_0': 'rec_TRANSITION', 'recall_1': 'rec_PURSUIT',
                          'recall_2': 'rec_REFIXATION', 'recall_3': 'rec_STABLE'})
show = ['bal_acc', 'macro_f1', 'auc_gate', 'rec_TRANSITION', 'rec_PURSUIT', 'rec_REFIXATION',
        'rec_STABLE', 'gate_false_skip_rate']
display(tab[show].round(3))
print('mean +/- std over folds:')
for c in show:
    print(f'   {c:<22} {tab[c].mean():.3f} +/- {tab[c].std():.3f}')
for f in res['folds']:
    v = f['vor_check']
    print(f"   fold {f['fold']}: IMU map {f['head_map_info'].get('mode')}, VOR median cos {v['median_cos']:+.3f}")

## 6. Ablations and the shuffle control

| run | flag | question |
|---|---|---|
| eye only | `--streams eye` | how far does gaze alone get in this model? |
| **IMU only** | `--streams head` | how far does head motion alone get, with no eye signal? |
| eye + head | `--streams eye,head` | are the two sensors complementary? |
| full | (section 5) | what does cos(eye, head) add on top? |
| shuffle control | `--shuffle_control` | features paired with the wrong labels: **must** fall to ≈ 0.25 balanced accuracy, or the setup leaks |

All variants train on the same rows. Only the last lines of each run are shown. A variant
whose `cv_results.json` already exists on Drive is skipped, so re-running this cell only
trains what is new; delete its folder under `MyDrive/GazeVLM/runs/` to retrain it.

In [ ]:
VARIANTS = {
    'eye_only':  '--streams eye',
    'head_only': '--streams head',          # IMU only: head speed, no eye signal at all
    'eye_head':  '--streams eye,head',
    'shuffle':   '--shuffle_control',
}
for name, flag in VARIANTS.items():
    out = os.path.join(RUNS, f'eyehead_{name}')
    if os.path.exists(os.path.join(out, 'cv_results.json')):
        print(f'{name}: already trained -> {out}   (delete the folder to retrain)')
        continue
    print(f"\n{'='*30} {name} {'='*30}")
    sh(f'python -u -m eyehead.train {COMMON} {flag} --out_dir "{out}"', tail=14)

rows = {}
for name, path in [('eye_only',  os.path.join(RUNS, 'eyehead_eye_only')),
                   ('head_only (IMU)', os.path.join(RUNS, 'eyehead_head_only')),
                   ('eye_head',  os.path.join(RUNS, 'eyehead_eye_head')),
                   ('full',      RUN_MAIN),
                   ('shuffle',   os.path.join(RUNS, 'eyehead_shuffle'))]:
    r = json.load(open(os.path.join(path, 'cv_results.json')))
    L = pd.DataFrame([f['last'] for f in r['folds']])
    rows[name] = {'bal_acc': f"{L.bal_acc.mean():.3f} +/- {L.bal_acc.std():.3f}",
                  'macro_f1': round(L.macro_f1.mean(), 3), 'auc_gate': round(L.auc_gate.mean(), 3),
                  'rec_TRANS': round(L.recall_0.mean(), 3),
                  'rec_PURSUIT': round(L.recall_1.mean(), 3), 'rec_REFIX': round(L.recall_2.mean(), 3),
                  'rec_STABLE': round(L.recall_3.mean(), 3),
                  'false_skip': round(L.gate_false_skip_rate.mean(), 3)}
cmp = pd.DataFrame(rows).T
display(cmp)

sh_bal = float(cmp.loc['shuffle', 'bal_acc'].split()[0])
print(f"shuffle control balanced accuracy {sh_bal:.3f}  "
      f"[{'PASS: near chance' if sh_bal < 0.32 else 'FAIL: the control learned something -- look for a leak'}]")
print('eye_only vs head_only: which single sensor carries more;  eye_head vs either: '
      'whether they are complementary.')

## 7. Test, once

The five fold models are averaged. Each scores the test rows with its own IMU map and
normalisation. The script prints per-model and ensemble metrics, the confusion matrix, the
skip-threshold sweep and, if the previous gate's `test_preds.csv` is on Drive, a
side-by-side on the same rows.

In [ ]:
PRED = os.path.join(RUN_MAIN, 'test_preds.csv')
base_flag = f'--baseline_preds "{BASELINE}"' if os.path.exists(BASELINE) else ''
sh(f'python -u -m eyehead.infer --folds_dir "{FOLDS}" --thresholds "{THRESH}" '
   f'--ckpt_dir "{RUN_MAIN}" --out_csv "{PRED}" {base_flag}')

### The test result in pictures

In [ ]:
pr = pd.read_csv(PRED)
summ = json.load(open(os.path.splitext(PRED)[0] + '_summary.json'))
labels = list(NAMES.values())
cm = np.array(summ['confusion'])
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for a_, mat, title, fmt in ((ax[0], cm, 'counts', '{:,.0f}'),
                            (ax[1], cm_norm, 'row-normalised (diagonal = recall)', '{:.2f}')):
    a_.imshow(mat, cmap='Blues')
    a_.set_xticks(range(4)); a_.set_xticklabels(labels, rotation=30, ha='right')
    a_.set_yticks(range(4)); a_.set_yticklabels(labels)
    for i in range(4):
        for j in range(4):
            a_.text(j, i, fmt.format(mat[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if mat[i, j] > mat.max() / 2 else 'black')
    a_.set_xlabel('predicted'); a_.set_ylabel('truth'); a_.set_title(title)

ens = summ['ensemble']
new_rec = [ens[f'recall_{c}'] for c in range(4)]
x = np.arange(4)
if summ.get('baseline'):
    old_rec = [summ['baseline'][f'recall_{c}'] for c in range(4)]
    ax[2].bar(x - .19, old_rec, .38, label='previous gaze-only gate', color='tab:gray')
    ax[2].bar(x + .19, new_rec, .38, label='eye + head', color='tab:blue')
else:
    ax[2].bar(x, new_rec, .5, label='eye + head', color='tab:blue')
ax[2].set_xticks(x); ax[2].set_xticklabels(labels, rotation=30, ha='right')
ax[2].set_ylim(0, 1); ax[2].set_title('test recall per class'); ax[2].legend()
plt.tight_layout(); plt.show()

sw = pd.DataFrame(summ['skip_sweep'])
plt.figure(figsize=(7, 5))
plt.plot(sw.skip_pct, sw.false_skip_pct, 'o-', label='eye + head, skip iff P(STABLE) > t')
plt.scatter([100 * (1 - ens['gate_send_rate'])], [100 * ens['gate_false_skip_rate']],
            s=90, c='tab:blue', marker='*', zorder=5, label='eye + head, argmax')
if summ.get('baseline'):
    b = summ['baseline']
    plt.scatter([100 * (1 - b['gate_send_rate'])], [100 * b['gate_false_skip_rate']],
                s=90, c='tab:gray', marker='s', zorder=5, label='previous gate')
oracle = 100 * (1 - ens['gate_oracle_send_rate'])
plt.axvline(oracle, ls=':', c='k', lw=1, label=f'oracle skip rate {oracle:.0f}%')
plt.xlabel('frames skipped (%)  -> compute saved'); plt.ylabel('false skips (%)  -> content lost')
plt.title('the gate trade-off: lower-right is better'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.show()

print(f"balanced accuracy {ens['bal_acc']:.3f}   macro F1 {ens['macro_f1']:.3f}   AUC_gate {ens['auc_gate']:.4f}")
print(f"REFIXATION recall {ens['recall_2']:.3f}  precision {ens['precision_2']:.3f}   "
      f"PURSUIT recall {ens['recall_1']:.3f}  precision {ens['precision_1']:.3f}")
print(f"skips {100*(1-ens['gate_send_rate']):.1f}%   false skip {100*ens['gate_false_skip_rate']:.1f}%")

## 8. All four labels: old gaze-only gate vs every new model

Five models on the same 2,285 test rows:

| model | input |
|---|---|
| **old gate** | gaze rates → predicted `[S_frame, S_gaze]` (the previous `gatetrain` model) |
| eye only | eye speed |
| **IMU only** | head speed from the gyro, no eye signal |
| eye + IMU | eye speed + head speed |
| full | eye speed + head speed + cos(eye, head) |

For each class: **AUC, recall, precision, F1**. Then the gate view (skip rate, false skips,
false sends) and ROC curves.

**How the old gate gets a per-class AUC.** It never output class probabilities, so each
class is scored by the signed margin on the weaker axis of its own quadrant, at the
thresholds that made the labels:

| class | quadrant | old-gate score |
|---|---|---|
| TRANSITION | frame low, gaze low | min(τ_f − S_frame, τ_g − S_gaze) |
| PURSUIT | frame low, gaze high | min(τ_f − S_frame, S_gaze − τ_g) |
| REFIXATION | frame high, gaze low | min(S_frame − τ_f, τ_g − S_gaze) |
| STABLE | frame high, gaze high | min(S_frame − τ_f, S_gaze − τ_g) |

Its STABLE AUC must equal its old AUC_gate (0.868); that is the check the scoring is
consistent. Its recall / precision / F1 come from its hard `pred_quad` labels.

**Intervals.** Each model's AUC gain over the old gate gets a 95% interval from a paired
bootstrap over the 12 test **videos** (rows within a video are correlated, so resampling
rows would make the interval far too narrow). An interval that excludes 0 means the gain is
not just which videos landed in test.

Section 6 already chose between the variants on cross-validation. Scoring them on test here
is for reporting; do not use these numbers to pick a model.

In [ ]:
from src.foldtrain.metrics import roc_auc
from eyehead.metrics import evaluate

assert os.path.exists(BASELINE), f'previous gate predictions not found: {BASELINE}'
classes = list(NAMES.values())
INV = {v: k for k, v in NAMES.items()}

# ---- test predictions for every new model (the full model's came from section 7) -------
NEW_MODELS = {
    'eye only':  os.path.join(RUNS, 'eyehead_eye_only'),
    'IMU only':  os.path.join(RUNS, 'eyehead_head_only'),
    'eye + IMU': os.path.join(RUNS, 'eyehead_eye_head'),
    'full':      RUN_MAIN,
}
for name, d in NEW_MODELS.items():
    assert os.path.exists(os.path.join(d, 'fold1.pt')), f'{name}: not trained yet -- run section 6'
    p = os.path.join(d, 'test_preds.csv')
    if not os.path.exists(p):
        print(f'scoring {name} on test ...')
        sh(f'python -u -m eyehead.infer --folds_dir "{FOLDS}" --thresholds "{THRESH}" '
           f'--ckpt_dir "{d}" --out_csv "{p}"', tail=2)

ref = pd.read_csv(os.path.join(RUN_MAIN, 'test_preds.csv'))
y, seq = ref.true_quad.map(INV).to_numpy(), ref.sequence.to_numpy()

# ---- per model: class scores (for AUC) and hard predictions (for recall/precision/F1) ---
old = pd.read_csv(BASELINE)
assert len(old) == len(ref) and (old.sequence.to_numpy() == seq).all(), 'old preds not row-aligned'
assert (old.true_quad.map(INV).to_numpy() == y).all(), 'old preds have different labels'
tf, tg = th['tau_frame'], th['tau_gaze']
f = old.pred_S_frame.to_numpy() - tf
g = old.pred_S_gaze.to_numpy() - tg
SCORES = {'old gate': np.stack([np.minimum(-f, -g), np.minimum(-f, g),
                                np.minimum(f, -g), np.minimum(f, g)], 1)}
HARD = {'old gate': old.pred_quad.map(INV).to_numpy()}
for name, d in NEW_MODELS.items():
    df_ = pd.read_csv(os.path.join(d, 'test_preds.csv'))
    assert (df_.sequence.to_numpy() == seq).all(), f'{name} preds not row-aligned'
    SCORES[name] = df_[[f'p_{c}' for c in classes]].to_numpy()
    HARD[name] = SCORES[name].argmax(1)
MODELS = list(SCORES)

# ---- per-class metrics -------------------------------------------------------------------
per = {}
for m in MODELS:
    met, _ = evaluate(np.eye(4)[HARD[m]], y)         # recall / precision / F1 / gate, from hard labels
    for k, c in enumerate(classes):
        met[f'auc_{k}'] = roc_auc(y == k, SCORES[m][:, k])   # AUC from the continuous scores
    met['macro_auc'] = float(np.mean([met[f'auc_{k}'] for k in range(4)]))
    per[m] = met

for metric, label in (('auc', 'AUC'), ('recall', 'recall'),
                      ('precision', 'precision'), ('f1', 'F1')):
    t = pd.DataFrame({m: [per[m][f'{metric}_{k}'] for k in range(4)] for m in MODELS},
                     index=classes)
    t.loc['mean'] = t.mean()
    print(f'\n{label} per class')
    display(t.round(3).style.highlight_max(axis=1, color='#cfe8ff'))

summ_tab = pd.DataFrame({m: {
    'balanced acc': per[m]['bal_acc'], 'macro F1': per[m]['macro_f1'],
    'macro AUC': per[m]['macro_auc'],
    'skip %': 100 * (1 - per[m]['gate_send_rate']),
    'false skip %': 100 * per[m]['gate_false_skip_rate'],
    'false send %': 100 * per[m]['gate_false_send_rate']} for m in MODELS}).T
print('\nsummary (gate skips STABLE only; oracle skip rate '
      f"{100*(1-per['full']['gate_oracle_send_rate']):.1f}%)")
display(summ_tab.round(3))

# ---- paired bootstrap over videos: AUC gain of each new model over the old gate ---------
B = 300
rng = np.random.default_rng(0)
vids = np.unique(seq)
by_vid = {v: np.flatnonzero(seq == v) for v in vids}
gains = {(m, k): [] for m in MODELS[1:] for k in range(4)}
for _ in range(B):
    idx = np.concatenate([by_vid[v] for v in rng.choice(vids, len(vids), replace=True)])
    for k in range(4):
        a_old = roc_auc(y[idx] == k, SCORES['old gate'][idx, k])
        for m in MODELS[1:]:
            gains[(m, k)].append(roc_auc(y[idx] == k, SCORES[m][idx, k]) - a_old)
ci = {}
for m in MODELS[1:]:
    ci[m] = []
    for k in range(4):
        d = np.array(gains[(m, k)]); d = d[np.isfinite(d)]
        lo, hi = np.percentile(d, [2.5, 97.5])
        ci[m].append(f"{per[m][f'auc_{k}'] - per['old gate'][f'auc_{k}']:+.3f}  [{lo:+.3f}, {hi:+.3f}]")
print(f'\nAUC gain over the old gate, with 95% interval (paired bootstrap, '
      f'{len(vids)} videos, {B} resamples)')
display(pd.DataFrame(ci, index=classes))
print('check: old gate STABLE AUC %.4f should equal its AUC_gate (0.868)'
      % per['old gate']['auc_3'])

# ---- plots -------------------------------------------------------------------------------
COL = {'old gate': 'tab:gray', 'eye only': 'tab:orange', 'IMU only': 'tab:green',
       'eye + IMU': 'tab:purple', 'full': 'tab:blue'}

def roc(t, s):
    o = np.argsort(-s); t = t[o].astype(int)
    tp, fp = np.cumsum(t), np.cumsum(1 - t)
    return np.r_[0, fp / fp[-1]], np.r_[0, tp / tp[-1]]

fig, ax = plt.subplots(1, 4, figsize=(21, 4.8))
for k, (a_, c) in enumerate(zip(ax, classes)):
    for m in MODELS:
        fx, ty = roc(y == k, SCORES[m][:, k])
        a_.plot(fx, ty, lw=1.8, color=COL[m], label=f"{m}  {per[m][f'auc_{k}']:.3f}")
    a_.plot([0, 1], [0, 1], 'k--', lw=1)
    a_.set_title(c); a_.set_xlabel('false positive rate'); a_.legend(fontsize=7, loc='lower right')
ax[0].set_ylabel('true positive rate')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 4, figsize=(21, 4.2), sharey=True)
x = np.arange(4); w = 0.8 / len(MODELS)
for a_, (metric, label) in zip(ax, (('auc', 'AUC'), ('recall', 'recall'),
                                     ('precision', 'precision'), ('f1', 'F1'))):
    for i, m in enumerate(MODELS):
        a_.bar(x + (i - (len(MODELS) - 1) / 2) * w,
               [per[m][f'{metric}_{k}'] for k in range(4)], w, color=COL[m], label=m)
    a_.set_xticks(x); a_.set_xticklabels(classes, rotation=25, ha='right')
    a_.set_ylim(0, 1); a_.set_title(f'{label} per class')
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Alternative policy: discard PURSUIT + STABLE, send TRANSITION + REFIXATION

Same five models, same test rows, one change in what the gate does:

| class | section 8 policy | this policy |
|---|---|---|
| TRANSITION | send | send |
| PURSUIT | send | **discard** |
| REFIXATION | send | send |
| STABLE | discard | discard |

PURSUIT and STABLE are exactly the two **gaze-high** quadrants, so under this policy the
ground truth is just `gaze_patch_token_sim > τ_gaze`; frame similarity drops out. So:

- **new models:** discard score = P(PURSUIT) + P(STABLE); hard decision = argmax ∈ {PURSUIT, STABLE}
- **old gate:** discard score = predicted S_gaze − τ_gaze; hard decision = its `pred_quad` ∈ {PURSUIT, STABLE}

| metric | meaning |
|---|---|
| **AUC (discard)** | ranking quality, independent of how often a model predicts PURSUIT; the fairest single number |
| correct discards % | of the frames that should be discarded, how many were |
| correct sends % | of the frames that should be sent, how many were |
| **false discard %** | content lost (of all frames) |
| false send % | compute wasted (of all frames) |

Requires section 8 to have run in this session (it reuses `y`, `SCORES`, `HARD`, `old`).

In [ ]:
DISC = [INV['PURSUIT'], INV['STABLE']]
truth = np.isin(y, DISC)
print(f"ground truth: {truth.sum():,} of {len(truth):,} frames should be discarded "
      f"({100*truth.mean():.1f}%), {(~truth).sum():,} sent")

DSCORE = {m: (old.pred_S_gaze.to_numpy() - tg) if m == 'old gate'
          else SCORES[m][:, DISC].sum(1) for m in MODELS}
DPRED = {m: np.isin(HARD[m], DISC) for m in MODELS}

rows = {}
for m in MODELS:
    p = DPRED[m]
    rows[m] = {'AUC (discard)':       roc_auc(truth, DSCORE[m]),
               'discard %':           100 * p.mean(),
               'correct discards %':  100 * (p & truth).sum() / truth.sum(),
               'discard precision %': 100 * (p & truth).sum() / max(1, p.sum()),
               'correct sends %':     100 * (~p & ~truth).sum() / (~truth).sum(),
               'false discard %':     100 * (p & ~truth).mean(),
               'false send %':        100 * (~p & truth).mean(),
               'agreement %':         100 * (p == truth).mean()}
pol = pd.DataFrame(rows).T
display(pol.round(3).style
        .highlight_max(subset=['AUC (discard)', 'correct discards %', 'correct sends %',
                               'agreement %'], color='#cfe8ff')
        .highlight_min(subset=['false discard %', 'false send %'], color='#cfe8ff'))

# where the lost content comes from: sent-classes that each model discarded
lost = pd.DataFrame({m: {c: int((DPRED[m] & (y == INV[c])).sum())
                         for c in ('TRANSITION', 'REFIXATION')} for m in MODELS}).T
lost['total'] = lost.sum(1)
print('\nfalse discards by true class (frames that should have been sent):')
display(lost)

# paired video bootstrap on the discard AUC gain over the old gate
gain = {m: [] for m in MODELS[1:]}
for _ in range(B):
    idx = np.concatenate([by_vid[v] for v in rng.choice(vids, len(vids), replace=True)])
    a0 = roc_auc(truth[idx], DSCORE['old gate'][idx])
    for m in MODELS[1:]:
        gain[m].append(roc_auc(truth[idx], DSCORE[m][idx]) - a0)
print(f'\ndiscard-AUC gain over the old gate, 95% interval ({len(vids)} videos, {B} resamples):')
for m in MODELS[1:]:
    d = np.array(gain[m]); d = d[np.isfinite(d)]
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"   {m:<10} {pol.loc[m, 'AUC (discard)'] - pol.loc['old gate', 'AUC (discard)']:+.3f}"
          f"   [{lo:+.3f}, {hi:+.3f}]")

# trade-off: sweep the discard threshold for each new model, mark the argmax points
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for m in MODELS:
    fx, ty = roc(truth, DSCORE[m])
    ax[0].plot(fx, ty, lw=1.8, color=COL[m], label=f"{m}  {pol.loc[m, 'AUC (discard)']:.3f}")
    s = DSCORE[m]
    qs = np.quantile(s, np.linspace(0.02, 0.98, 40))
    disc = [100 * (s > q).mean() for q in qs]
    fdisc = [100 * ((s > q) & ~truth).mean() for q in qs]
    ax[1].plot(disc, fdisc, lw=1.5, color=COL[m], alpha=.8)
    ax[1].scatter([pol.loc[m, 'discard %']], [pol.loc[m, 'false discard %']],
                  s=80, color=COL[m], edgecolor='k', zorder=5, label=f'{m} (argmax)')
ax[0].plot([0, 1], [0, 1], 'k--', lw=1)
ax[0].set_xlabel('false positive rate'); ax[0].set_ylabel('true positive rate')
ax[0].set_title('ROC: discard PURSUIT + STABLE'); ax[0].legend(fontsize=8, loc='lower right')
ax[1].axvline(100 * truth.mean(), ls=':', c='k', lw=1, label=f'oracle {100*truth.mean():.0f}%')
ax[1].set_xlabel('frames discarded (%)  -> compute saved')
ax[1].set_ylabel('false discards (%)  -> content lost')
ax[1].set_title('trade-off: lower-right is better'); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

---

## Reading it

| | |
|---|---|
| **balanced accuracy** | chance is 0.25. Read it before plain accuracy, which TRANSITION and STABLE dominate |
| **REFIXATION / PURSUIT recall** | the reason this model exists. The previous gate: 0.00 and 0.09 on test |
| **REFIXATION precision** | speed alone cannot tell "saccade and back" (STABLE) from "saccade to somewhere new" (REFIXATION); low precision here is that limitation |
| **false skip** | content lost, unrecoverable. The previous gate: 5.7% on test |
| **eye only vs eye + head** | what the IMU is worth |
| **eye + head vs full** | what cos(eye, head) is worth |
| **shuffle control** | must sit near 0.25. If not, stop and find the leak |
| **VOR sign check** | must be clearly negative in calib mode, or the head axes are wrong |

## If REFIXATION and PURSUIT stay low

1. Check section 4 first: if the classes overlap in the raw signals, the limit is the
   input, not the model.
2. Try `--class_weight sqrt` if the rare classes are over-predicted, or keep `inv` if they
   are still ignored.
3. Compare `--imu_rot calib` with `--imu_rot fit`; a wrong head map makes cos noise.
4. The labels themselves are DINOv2 cosines cut at the median, so many rows sit right at a
   class boundary. Some confusion there is label noise, not model error.

## What is not tested here

Whether a VLM answers better on the frames this gate keeps. Every number above measures
agreement with a DINOv2 similarity proxy.